# CharWriteViT on Kaggle
OCR, Writer ID, Discriminator, and Character Classifier are trained jointly with the generator. Only `ngkinwang/iam-dataset` is required; character labels come directly from query-token-aligned decoder cells.

In [ ]:
REPOSITORY = 'https://github.com/YOUR_USER/YOUR_REPO.git'
!git clone $REPOSITORY /kaggle/working/CharWriteViT
%cd /kaggle/working/CharWriteViT
!pip install -q -e .

In [ ]:
# Inspect attached data. EMNIST is required for classifier pretraining; licensed fonts add handwriting punctuation.
!find /kaggle/input -maxdepth 4 -type f | head -80

In [ ]:
# Shared helpers. IAM training uses a separately pretrained, frozen character classifier.
from pathlib import Path
import os, subprocess, sys
from IPython.display import display, Image

PROJECT = Path('/kaggle/working/CharWriteViT')
# Reduces CUDA allocator fragmentation on Kaggle's 16 GB GPUs. Each training command runs in a fresh subprocess.
TRAIN_ENV = os.environ.copy()
TRAIN_ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Pretrain the character classifier before starting IAM training.')

In [ ]:
# Pretrain the corrected classifier once: uncropped EMNIST plus an explicit GNU anchor.
CLASSIFIER_OUTPUT = PROJECT / 'outputs' / 'classifier-v2'
CHARACTER_CLASSIFIER_PRETRAIN = CLASSIFIER_OUTPUT / 'best.pt'
if not CHARACTER_CLASSIFIER_PRETRAIN.is_file():
    subprocess.run([sys.executable, '-m', 'charwritevit.cli.train_classifier',
                    '--config', str(PROJECT / 'configs/kaggle.yaml'),
                    '--output', str(CLASSIFIER_OUTPUT)],
                   cwd=PROJECT, env=TRAIN_ENV, check=True)
print('Frozen classifier:', CHARACTER_CLASSIFIER_PRETRAIN)
display(Image(filename=str(CLASSIFIER_OUTPUT / 'classifier_curves.png')))
gnu_previews = sorted((CLASSIFIER_OUTPUT / 'preview').glob('*_gnu.png'))
validation_previews = sorted((CLASSIFIER_OUTPUT / 'preview').glob('*_validation.png'))
display(Image(filename=str(gnu_previews[-1])))
display(Image(filename=str(validation_previews[-1])))

In [ ]:
# Prepare IAM after attaching ngkinwang/iam-dataset.
subprocess.run([sys.executable, '-m', 'charwritevit.cli.prepare_iam',
                '--config', str(PROJECT / 'configs/kaggle.yaml')],
               cwd=PROJECT, check=True)
print((PROJECT / 'configs/kaggle.yaml').read_text())
print((Path('/kaggle/working/charwritevit-data/iam') / 'summary.json').read_text())

In [ ]:
# End-to-end IAM smoke test with the frozen pretrained character classifier.
IAM_SMOKE = PROJECT / 'outputs' / 'iam-smoke'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.train_iam',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--output', str(IAM_SMOKE), '--classifier', str(CHARACTER_CLASSIFIER_PRETRAIN),
                '--epochs', '1', '--max-steps', '2'],
               cwd=PROJECT, env=TRAIN_ENV, check=True)
for name in ['sample_00_characters.png', 'sample_00_scales.png', 'sample_00_word.png']:
    display(Image(filename=str(IAM_SMOKE / 'preview' / 'epoch_000' / name)))

In [ ]:
# Full IAM training. Use a fresh output because runs guided by the broken classifier cannot be resumed.
IAM_OUTPUT = PROJECT / 'outputs' / 'iam-classifier-v2'
command = [sys.executable, '-m', 'charwritevit.cli.train_iam',
           '--config', str(PROJECT / 'configs/kaggle.yaml'),
           '--output', str(IAM_OUTPUT),
           '--classifier', str(CHARACTER_CLASSIFIER_PRETRAIN)]
latest = IAM_OUTPUT / 'latest.pt'
if latest.is_file(): command += ['--resume', str(latest)]
subprocess.run(command, cwd=PROJECT, env=TRAIN_ENV, check=True)

In [ ]:
# Evaluate the final checkpoint, including standard Inception FID and KID.
EVAL_OUTPUT = IAM_OUTPUT / 'evaluation'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.evaluate',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--checkpoint', str(IAM_OUTPUT / 'latest.pt'),
                '--output', str(EVAL_OUTPUT),
                '--max-batches', '100', '--kid-subset-size', '100'],
               cwd=PROJECT, check=True)
print((EVAL_OUTPUT / 'metrics.json').read_text())

In [ ]:
# Display loss curve and all 10 samples from the newest completed epoch.
display(Image(filename=str(IAM_OUTPUT / 'iam_curves.png')))
preview_epochs = sorted((IAM_OUTPUT / 'preview').glob('epoch_*'))
latest_preview = preview_epochs[-1]
for sample in range(10):
    for suffix in ('characters', 'scales', 'word'):
        artifact = latest_preview / f'sample_{sample:02d}_{suffix}.png'
        if artifact.is_file(): display(Image(filename=str(artifact)))

In [ ]:
# Archive checkpoints so the notebook Output can be saved as a Kaggle Dataset.
!tar -czf /kaggle/working/charwritevit-checkpoints.tgz outputs configs/kaggle.yaml